In [4]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [5]:
!pip install -q kaggle
!kaggle datasets download -d puneet6060/intel-image-classification -p data/
!cd data && unzip -q intel-image-classification.zip && rm intel-image-classification.zip

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:21<00:00, 17.0MB/s]



In [6]:
!ls data
!ls data/seg_train/seg_train

seg_pred  seg_test  seg_train
buildings  forest  glacier  mountain  sea  street


In [7]:
import os
from collections import Counter

import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib
matplotlib.use("Agg")  # без графического окна — просто сохраняем картинки в файл
import matplotlib.pyplot as plt

In [8]:
TRAIN_DIR = os.path.join("data", "seg_train", "seg_train")
TEST_DIR = os.path.join("data", "seg_test", "seg_test")

IMG_SIZE = 150          # изображения приведём к 150x150 (исходный размер датасета)
BATCH_SIZE = 32         # сколько картинок в одной "пачке" при обучении
VAL_RATIO = 0.2         # 20% обучающих данных отдаём под валидацию

In [9]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # одинаковый размер для всех картинок
    transforms.ToTensor(),                    # PIL-изображение -> тензор [0,1], форма (C,H,W)
])

# Базовый train-трансформ (БЕЗ аугментации) — пригодится для сравнения на Неделе 3.
train_transform_plain = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [10]:
def get_datasets():
    """Возвращает (train, val, test) как объекты ImageFolder/Subset.

    ImageFolder сам понимает классы по именам подпапок и присваивает им метки.
    """
    # Полный обучающий набор; разобьём его на train и val.
    full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_transform_plain)
    test_set = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

    # Считаем размеры частей и делаем воспроизводимое разбиение (фиксируем seed).
    val_size = int(len(full_train) * VAL_RATIO)
    train_size = len(full_train) - val_size
    generator = torch.Generator().manual_seed(42)  # seed -> одинаковое разбиение каждый раз
    train_set, val_set = random_split(full_train, [train_size, val_size], generator=generator)

    return train_set, val_set, test_set, full_train.classes


def get_dataloaders():
    """Оборачивает наборы данных в DataLoader (выдаёт данные пачками)."""
    train_set, val_set, test_set, classes = get_datasets()
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)   # перемешиваем train
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader, classes

In [11]:
# --- EDA: анализ данных ------------------------------------------------------
def run_eda():
    """Считает картинки по классам и сохраняет сетку примеров в results/."""
    os.makedirs("results", exist_ok=True)

    # Загружаем train БЕЗ преобразований, чтобы видеть оригинальные картинки.
    raw = datasets.ImageFolder(TRAIN_DIR)
    classes = raw.classes
    print("Классы:", classes)

    # raw.targets — список меток-классов для каждой картинки. Counter их посчитает.
    counts = Counter(raw.targets)
    print("\nКоличество изображений по классам (train):")
    for idx, name in enumerate(classes):
        print(f"  {name:10s}: {counts[idx]}")

    # Сохраняем по одному примеру каждого класса в одну картинку-сетку.
    fig, axes = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3))
    for idx, name in enumerate(classes):
        # находим первую картинку, у которой метка == idx
        sample_pos = raw.targets.index(idx)
        image, _ = raw[sample_pos]
        axes[idx].imshow(image)
        axes[idx].set_title(name)
        axes[idx].axis("off")
    fig.tight_layout()
    out_path = os.path.join("results", "class_samples.png")
    fig.savefig(out_path, dpi=120)
    print(f"\nСетка примеров сохранена в: {out_path}")


if __name__ == "__main__":
    run_eda()


Классы: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

Количество изображений по классам (train):
  buildings : 2191
  forest    : 2271
  glacier   : 2404
  mountain  : 2512
  sea       : 2274
  street    : 2382

Сетка примеров сохранена в: results/class_samples.png


In [12]:
!mkdir -p src
!mv model.py engine.py train_baseline.py src/ 2>/dev/null
!mv data_loading.py src/ 2>/dev/null
!ls src/

In [16]:
%cd /content
!unzip -o "intel-dl-project-week2[1].zip"     # кавычки обязательны из-за скобок в имени
!mkdir -p src
!cp intel-dl-project/src/*.py src/            # копируем все 4 .py с правильными именами
!ls src/

/content
Archive:  intel-dl-project-week2[1].zip
   creating: intel-dl-project/
  inflating: intel-dl-project/.gitignore  
  inflating: intel-dl-project/README.md  
   creating: intel-dl-project/data/
  inflating: intel-dl-project/data/README.md  
   creating: intel-dl-project/notebooks/
  inflating: intel-dl-project/proposal.md  
   creating: intel-dl-project/reports/
  inflating: intel-dl-project/reports/week-01.md  
  inflating: intel-dl-project/reports/week-02.md  
  inflating: intel-dl-project/requirements.txt  
   creating: intel-dl-project/results/
   creating: intel-dl-project/src/
  inflating: intel-dl-project/src/data_loading.py  
  inflating: intel-dl-project/src/engine.py  
  inflating: intel-dl-project/src/model.py  
  inflating: intel-dl-project/src/train_baseline.py  
data_loading.py  engine.py  model.py  train_baseline.py


In [17]:
!python src/train_baseline.py

Устройство: cuda
Классы: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Эпоха  1/10 | train loss 1.160 acc 0.530 | val loss 0.883 acc 0.614
Эпоха  2/10 | train loss 0.890 acc 0.659 | val loss 0.777 acc 0.700
Эпоха  3/10 | train loss 0.748 acc 0.723 | val loss 0.747 acc 0.713
Эпоха  4/10 | train loss 0.660 acc 0.767 | val loss 0.628 acc 0.760
Эпоха  5/10 | train loss 0.581 acc 0.794 | val loss 0.541 acc 0.803
Эпоха  6/10 | train loss 0.550 acc 0.805 | val loss 0.492 acc 0.819
Эпоха  7/10 | train loss 0.520 acc 0.818 | val loss 0.479 acc 0.828
Эпоха  8/10 | train loss 0.503 acc 0.821 | val loss 0.495 acc 0.825
Эпоха  9/10 | train loss 0.469 acc 0.830 | val loss 0.457 acc 0.837
Эпоха 10/10 | train loss 0.440 acc 0.842 | val loss 0.502 acc 0.816
Графики сохранены в: results/baseline_curves.png

Test: loss 0.471 | accuracy 0.829

              precision    recall  f1-score   support

   buildings      0.795     0.854     0.823       437
      forest      0.944     0.962    

In [18]:
!python src/train_baseline.py      # заново, чтобы появилась сводка (быстро)
!python src/train_augmented.py     # baseline + аугментация
!python src/train_resnet.py        # ResNet-18 (чуть дольше — картинки 224x224)
!python src/compare.py             # печатает таблицу сравнения

Устройство: cuda
Классы: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Эпоха  1/10 | train loss 1.160 acc 0.533 | val loss 0.867 acc 0.634
Эпоха  2/10 | train loss 0.889 acc 0.663 | val loss 0.760 acc 0.708
Эпоха  3/10 | train loss 0.744 acc 0.728 | val loss 0.632 acc 0.773
Эпоха  4/10 | train loss 0.654 acc 0.769 | val loss 0.601 acc 0.772
Эпоха  5/10 | train loss 0.577 acc 0.795 | val loss 0.542 acc 0.803
Эпоха  6/10 | train loss 0.560 acc 0.801 | val loss 0.505 acc 0.809
Эпоха  7/10 | train loss 0.528 acc 0.815 | val loss 0.479 acc 0.826
Эпоха  8/10 | train loss 0.504 acc 0.818 | val loss 0.490 acc 0.824
Эпоха  9/10 | train loss 0.472 acc 0.833 | val loss 0.470 acc 0.826
Эпоха 10/10 | train loss 0.445 acc 0.840 | val loss 0.477 acc 0.820
Графики сохранены в: results/baseline_curves.png

Test: loss 0.471 | accuracy 0.828

              precision    recall  f1-score   support

   buildings      0.760     0.883     0.817       437
      forest      0.922     0.975    

In [26]:
%cd /content
from google.colab import files
files.upload()        # выбери intel-dl-project-week3.zip

!unzip -o intel-dl-project-week3*.zip      # звёздочка на случай суффикса [1]
!cp -f intel-dl-project/src/*.py src/      # перезаписываем src новыми версиями
!ls src/

/content


Saving intel-dl-project-week3[1].zip to intel-dl-project-week3[1].zip
Archive:  intel-dl-project-week3[1].zip
  inflating: intel-dl-project/.gitignore  
  inflating: intel-dl-project/README.md  
  inflating: intel-dl-project/data/README.md  
  inflating: intel-dl-project/proposal.md  
  inflating: intel-dl-project/reports/week-01.md  
  inflating: intel-dl-project/reports/week-02.md  
  inflating: intel-dl-project/reports/week-03.md  
  inflating: intel-dl-project/requirements.txt  
  inflating: intel-dl-project/src/compare.py  
  inflating: intel-dl-project/src/data_loading.py  
  inflating: intel-dl-project/src/engine.py  
  inflating: intel-dl-project/src/model.py  
  inflating: intel-dl-project/src/train_augmented.py  
  inflating: intel-dl-project/src/train_baseline.py  
  inflating: intel-dl-project/src/train_resnet.py  
compare.py	 engine.py  __pycache__		train_baseline.py
data_loading.py  model.py   train_augmented.py	train_resnet.py


In [27]:
!python src/train_baseline.py      # ~пара минут
!python src/train_augmented.py     # baseline + аугментация
!python src/train_resnet.py        # ResNet-18, чуть дольше (картинки 224x224)
!python src/compare.py             # печатает таблицу сравнения

Устройство: cuda
Классы: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Эпоха  1/10 | train loss 1.163 acc 0.530 | val loss 0.866 acc 0.635
Эпоха  2/10 | train loss 0.881 acc 0.666 | val loss 0.705 acc 0.731
Эпоха  3/10 | train loss 0.729 acc 0.734 | val loss 0.660 acc 0.751
Эпоха  4/10 | train loss 0.651 acc 0.770 | val loss 0.613 acc 0.777
Эпоха  5/10 | train loss 0.585 acc 0.792 | val loss 0.533 acc 0.805
Эпоха  6/10 | train loss 0.562 acc 0.798 | val loss 0.507 acc 0.817
Эпоха  7/10 | train loss 0.523 acc 0.817 | val loss 0.482 acc 0.823
Эпоха  8/10 | train loss 0.500 acc 0.823 | val loss 0.521 acc 0.814
Эпоха  9/10 | train loss 0.469 acc 0.834 | val loss 0.483 acc 0.825
Эпоха 10/10 | train loss 0.448 acc 0.841 | val loss 0.460 acc 0.826
Графики сохранены в: results/baseline_curves.png

[Baseline CNN (no augmentation)] test accuracy = 0.846 | macro F1 = 0.848

              precision    recall  f1-score   support

   buildings      0.831     0.858     0.845       4

In [23]:
%cd /content
from google.colab import files
files.upload()        # выбери intel-dl-project-week4-FINAL.zip
!unzip -o intel-dl-project-week4*.zip
!cp -f intel-dl-project/src/*.py src/
!ls src

/content


Saving intel-dl-project-week4-FINAL.zip to intel-dl-project-week4-FINAL.zip
Archive:  intel-dl-project-week4-FINAL.zip
  inflating: intel-dl-project/.gitignore  
  inflating: intel-dl-project/README.md  
  inflating: intel-dl-project/data/README.md  
  inflating: intel-dl-project/final-report.md  
  inflating: intel-dl-project/presentation_outline.md  
  inflating: intel-dl-project/proposal.md  
  inflating: intel-dl-project/reports/week-01.md  
  inflating: intel-dl-project/reports/week-02.md  
  inflating: intel-dl-project/reports/week-03.md  
  inflating: intel-dl-project/reports/week-04.md  
  inflating: intel-dl-project/requirements.txt  
  inflating: intel-dl-project/src/compare.py  
  inflating: intel-dl-project/src/data_loading.py  
  inflating: intel-dl-project/src/engine.py  
  inflating: intel-dl-project/src/error_analysis.py  
  inflating: intel-dl-project/src/model.py  
  inflating: intel-dl-project/src/train_augmented.py  
  inflating: intel-dl-project/src/train_baseline.

In [24]:
!python src/train_resnet.py        # ~пара минут, создаст resnet18.pth
!python src/error_analysis.py      # анализ ошибок

Устройство: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 196MB/s]
Эпоха  1/5 | train loss 0.328 acc 0.887 | val loss 0.220 acc 0.922
Эпоха  2/5 | train loss 0.208 acc 0.924 | val loss 0.208 acc 0.928
Эпоха  3/5 | train loss 0.165 acc 0.940 | val loss 0.214 acc 0.926
Эпоха  4/5 | train loss 0.139 acc 0.949 | val loss 0.205 acc 0.932
Эпоха  5/5 | train loss 0.110 acc 0.960 | val loss 0.233 acc 0.926
Графики сохранены в: results/resnet_curves.png

[ResNet-18 (transfer learning)] test accuracy = 0.926 | macro F1 = 0.928

              precision    recall  f1-score   support

   buildings      0.898     0.947     0.922       437
      forest      0.992     0.992     0.992       474
     glacier      0.916     0.846     0.880       553
    mountain      0.863     0.910     0.886       525
         sea      0.945     0.976     0.960       510
      street      0.953     

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
